**Cell #01**

# RAG11 Nutrition — Stage 1.1: Extract & Chunk.

Lists the source PDFs directly from the public `GOOGLE_DRIVE_SOURCES_FOLDER`
Drive folder (no hardcoded file ids), pulls any that aren't already sitting
locally, extracts hierarchical sections using the per-source algorithm in
`./stage1_1_eda_packages/` (one Python module per source PDF, matched by
filename), and writes parent/child chunk JSON files, per the project
overview doc.

- Input PDFs land in `./stage1_eda_input/source{1,2,3}/` (unchanged from before)
- Chunk output lands in `./stage1_eda_output/source{1,2,3}/`
- A `rag11_data_sources`-ready row per source lands in `./stage1_eda_output/sources/`
  (loaded into Supabase by `stage1_2_eda_load_chunks.ipynb`, alongside
  `sql/create_sql_tables.sql`'s `rag11_data_sources` table)

Run cells top to bottom. The Drive folder listing, downloads, and
page-text extraction are all cached/idempotent, so re-running the notebook
after tuning the section-detection cells below does not re-list, re-download,
or re-parse anything that's already on disk.


In [1]:
# Cell #02
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


**Cell #03**

## Config — sources, paths

In [2]:
# Cell #04
from pathlib import Path
import json, re, uuid
from datetime import datetime, timezone

import fitz  # PyMuPDF
import gdown

from stage1_1_eda_packages import KNOWN_SOURCE_EDA_META, SKIPPED_FILENAMES

PROJECT_ROOT = Path(".").resolve()
INPUT_ROOT = PROJECT_ROOT / "stage1_eda_input"
OUTPUT_ROOT = PROJECT_ROOT / "stage1_eda_output"
SOURCES_MANIFEST_DIR = OUTPUT_ROOT / "sources"

# Cap per-source page-TEXT extraction (see extract_pages() below) so the
# whole pipeline -- every registered source, all the way through
# write_chunks() -- can be run and sanity-checked in minutes instead of
# the tens of minutes a full 1000+ page book (or OCR pass) takes. Section
# *boundaries* still come from each source's own outline/regex algorithm
# against the real full PDF (cheap -- no per-page text work), so this only
# limits how much of each section's actual TEXT is available; a section
# starting past the cap ends up with empty text. That's fine for exercising
# code paths, but it is NOT a substitute for a full real-page run before
# trusting a new/changed source's output. Set to None for a full run.
MAX_NUMBER_OF_PAGES_TO_USE = 100

# Public Google Drive folder holding the source PDFs. Kept as a plain
# constant -- no API key/credentials needed -- because the folder is shared
# "Anyone with the link"; gdown's folder-page parser lists it directly.
GOOGLE_DRIVE_SOURCES_FOLDER = "https://drive.google.com/drive/folders/1GwS2oNWkn_aLE1eDTbkHW73Ljun_aM4I?usp=drive_link"

# Same uuid5 namespace stage1_2/stage1_9 use for parent/child rowGUIDs, so a
# source's rowGUID is identical no matter which notebook computes it, and
# stable across every re-run (uuid5 is deterministic given the same inputs).
RAG11_UUID_NAMESPACE = uuid.uuid5(uuid.NAMESPACE_DNS, "rag11.nutrition.poc")


def deterministic_uuid(business_key: str) -> str:
    return str(uuid.uuid5(RAG11_UUID_NAMESPACE, business_key))


def _drive_folder_id(url: str) -> str:
    m = re.search(r"/folders/([a-zA-Z0-9_-]+)", url)
    if not m:
        raise ValueError(f"Could not find a Drive folder id in: {url}")
    return m.group(1)


DRIVE_FOLDER_ID = _drive_folder_id(GOOGLE_DRIVE_SOURCES_FOLDER)


def list_drive_folder_files(folder_id: str) -> list[dict]:
    """
    List every file sitting directly in the public Drive folder, WITHOUT
    downloading anything yet, using gdown's own folder-page parser.

    Returns one {"file_id", "name"} dict per file. Needs gdown >= 6.1 (adds
    `skip_download=`, returning `GoogleDriveFileToDownload(id, path,
    local_path)` records instead of downloaded file paths -- see
    requirements.txt). Falls back to a full folder download into a
    throwaway temp dir on older gdown, so this keeps working either way,
    just without the "list first, download later" shortcut.
    """
    try:
        records = gdown.download_folder(
            id=folder_id, skip_download=True, quiet=True, use_cookies=False,
        )
        files = [{"file_id": r.id, "name": Path(r.path).name} for r in (records or [])]
    except TypeError:
        import tempfile
        print("  [list_drive_folder_files] installed gdown has no skip_download= "
              "support; listing by downloading into a temp folder instead "
              "(upgrade gdown>=6.1 to skip this extra download).")
        with tempfile.TemporaryDirectory() as tmp:
            gdown.download_folder(id=folder_id, output=tmp, quiet=True, use_cookies=False)
            files = [{"file_id": None, "name": p.name}
                     for p in sorted(Path(tmp).rglob("*")) if p.is_file()]

    if not files:
        raise RuntimeError(
            f"No files found in Drive folder {folder_id}. Confirm "
            "GOOGLE_DRIVE_SOURCES_FOLDER is still shared as 'Anyone with the link'."
        )
    return files


# Per-file EDA metadata (expected_pages/structure) now lives in
# ./stage1_1_eda_packages/ -- KNOWN_SOURCE_EDA_META (imported above) is
# built there from each source module's own EXPECTED_PAGES/STRUCTURE, right
# next to the algorithm that was designed against that page count.
# Keyed by FILENAME (not file_id or Drive listing position), so it survives
# a file being re-uploaded to Drive under a new file_id, and so the Drive
# folder's own listing order can never silently swap which PDF is treated
# as "source1" vs "source2" (see _source_order_key below).

# Known sources keep their historical source1/2/3 slot regardless of what
# order Drive happens to list them in; anything new discovered in the
# folder is appended after them (sorted by name) as source4, source5, ...
# instead of shifting the three known ones around underneath already-tuned
# section-detection logic.
_KNOWN_SOURCE_ORDER = list(KNOWN_SOURCE_EDA_META.keys())


def _source_order_key(f: dict) -> tuple:
    try:
        return (0, _KNOWN_SOURCE_ORDER.index(f["name"]))
    except ValueError:
        return (1, f["name"])


_drive_files_all = sorted(list_drive_folder_files(DRIVE_FOLDER_ID), key=_source_order_key)

# Filter out any file known to be unusable (e.g. a broken/bogus download
# that isn't really the book -- see SKIPPED_FILENAMES / a module's own
# SKIP_REASON in stage1_1_eda_packages) before SOURCES is built at all,
# so a skipped file never gets a source-slot number, is never
# downloaded/paged, and never reaches FILENAME_PROCESSORS below.
drive_files = []
for f in _drive_files_all:
    reason = SKIPPED_FILENAMES.get(f["name"])
    if reason:
        print(f'  [skip] {f["name"]}: {reason}')
        continue
    drive_files.append(f)

print(f"Drive folder {DRIVE_FOLDER_ID}: {len(_drive_files_all)} file(s) found, "
      f"{len(drive_files)} usable -> "
      + ", ".join(f["name"] for f in drive_files))

# SOURCES is now built programmatically from the Drive folder listing above,
# instead of a hardcoded {file_id, filename, ...} dict. A file not covered
# by KNOWN_SOURCE_EDA_META still gets a source slot (expected_pages=None,
# structure="unknown") -- see FILENAME_PROCESSORS further down for what
# happens if nothing knows how to chunk it yet.
SOURCES = {}
for idx, f in enumerate(drive_files, start=1):
    key = f"source{idx}"
    meta = KNOWN_SOURCE_EDA_META.get(f["name"], {"expected_pages": None, "structure": "unknown"})
    SOURCES[key] = {
        "file_id": f["file_id"],
        "filename": f["name"],
        "expected_pages": meta["expected_pages"],
        "structure": meta["structure"],
        # This source's own row in rag11_data_sources. Also becomes
        # rowOwnerGUID on every rag11_chunks_parent_table / _child_table row
        # produced for it -- see write_chunks() and sql/create_sql_tables.sql.
        "row_guid": deterministic_uuid(f"source:{f['file_id'] or f['name']}"),
    }

for key in SOURCES:
    (INPUT_ROOT / key).mkdir(parents=True, exist_ok=True)
    (OUTPUT_ROOT / key).mkdir(parents=True, exist_ok=True)
SOURCES_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)


Drive folder 1GwS2oNWkn_aLE1eDTbkHW73Ljun_aM4I: 17 file(s) found, 17 usable -> human-nutrition-text.pdf, Nutrition_for_Nurses-WEB_260913_200839.pdf, Nutrition-Science-and-Everyday-Application-1773787282.pdf, Advanced_Nutrition_and_Human_Metabolism.pdf, Clark N. - Nancy Clark's Food Guide for New Runners. Getting It Right from the Start - 2009.pdf, Clark N. - Nancy Clark's Food Guide for New Runners. Getting It Right from the Start - 2009.pdf, Intuitive_Eating_A_Revolutionary_Program_that_Works.pdf, Krauses_Food_and_the_Nutrition_Care_Process.pdf, Medical_Nutrition_and_Disease_A_Case_Based_Approach.pdf, Sports-Nutrition-Laboratory-Manual-Mary-P.-Miles-Stephanie-M.G.-Wilson-and-Morgan-L.-Chamberlin.pdf, _OceanofPDF.com_Nancy_Clarks_Sports_Nutrition_Guidebook_-_Nancy_Clark.pdf, _OceanofPDF.com_Peak_-_Marc_Bubbs.pdf, vdoc.pub_medical-nutrition-and-disease-a-case-based-approach.pdf, vdoc.pub_motivational-interviewing-in-nutrition-and-fitness.pdf, vdoc.pub_nutrition-therapy-and-pathophysiolo

**Cell #05**

## Download (only if missing locally)

Idempotent: skips any source whose PDF is already sitting in
`stage1_eda_input/<source>/`. Uses `gdown`, which handles Google Drive's
large-file "can't scan for viruses" confirmation flow automatically. Runs across all sources concurrently
(a thread pool -- see `_fetch_one`/`ThreadPoolExecutor` below), since
each download is just an idle network wait.


In [3]:
# Cell #06
from concurrent.futures import ThreadPoolExecutor, as_completed


def download_if_missing(source_key: str) -> Path:
    """
    Ensure the source PDF exists locally at
    ./stage1_eda_input/<source_key>/<filename>, downloading it from Google
    Drive with gdown if it isn't there yet. Safe to re-run.
    """
    cfg = SOURCES[source_key]
    filename = cfg["filename"]
    file_id = cfg["file_id"]
    target = INPUT_ROOT / source_key / filename

    if target.exists() and target.stat().st_size > 0:
        print(f"[{source_key}] already present: {target} ({target.stat().st_size / 1e6:.1f} MB)")
        return target

    if not file_id:
        raise RuntimeError(
            f"[{source_key}] no Drive file_id available for {filename} (the "
            "installed gdown fell back to the no-skip_download listing path, "
            "which can't resolve individual file ids -- upgrade gdown>=6.1, "
            f"or download this file manually into {target})."
        )

    print(f"[{source_key}] downloading {filename} (Drive id {file_id}) -> {target}")
    # Pass id= directly (rather than a uc?id= URL) so this works across gdown
    # versions regardless of whether fuzzy= is supported.
    gdown.download(id=file_id, output=str(target), quiet=False)

    if not target.exists() or target.stat().st_size == 0:
        raise RuntimeError(
            f"[{source_key}] download failed or produced an empty file at {target}. "
            "If Drive shows a warning page instead of the PDF, re-run this cell, or "
            "download it manually into that path."
        )
    return target


def verify_page_count(source_key: str, path: Path) -> int:
    cfg = SOURCES[source_key]
    expected = cfg["expected_pages"]
    with fitz.open(path) as doc:
        n = doc.page_count
    if expected is None:
        print(f"[{source_key}] {n} pages (no expected-page baseline for this file)")
    else:
        flag = "OK" if n == expected else "MISMATCH"
        print(f"[{source_key}] {n} pages (expected {expected}) -> {flag}")
    return n


def _fetch_one(key: str) -> tuple[str, Path]:
    return key, download_if_missing(key)


# Downloads run in parallel -- gdown.download() is a blocking HTTP call
# per file, so with a dozen-plus multi-hundred-page PDFs this was the
# slowest part of a cold run purely from sitting idle on network I/O.
# Capped at 6 concurrent workers rather than one-per-source to stay
# polite to Drive's per-IP rate limiting on anonymous downloads --
# verify_page_count() (opens the PDF, CPU-bound but fast) still runs
# on the main thread right after each download finishes, not inside
# the pool.
downloaded_paths = {}
source_fetch_info = {}  # source_key -> fetch/verification metadata for rag11_data_sources
with ThreadPoolExecutor(max_workers=min(6, len(SOURCES) or 1)) as pool:
    futures = [pool.submit(_fetch_one, key) for key in SOURCES]
    for fut in as_completed(futures):
        key, p = fut.result()  # raises here if that source's download failed
        n_pages = verify_page_count(key, p)
        downloaded_paths[key] = p
        source_fetch_info[key] = {
            "local_path": str(p.relative_to(PROJECT_ROOT)),
            "size_bytes": p.stat().st_size,
            "actual_page_count": n_pages,
            "fetched_at": datetime.now(timezone.utc).isoformat(),
        }

# Downstream cells iterate `for key in SOURCES` (never `downloaded_paths`
# directly), so the fact that downloads complete out of order here has no
# effect on section/chunk ordering.


[source1] already present: /Users/mgtimber/CV26/RAG11/stage1_eda_input/source1/human-nutrition-text.pdf (26.9 MB)[source2] already present: /Users/mgtimber/CV26/RAG11/stage1_eda_input/source2/Nutrition_for_Nurses-WEB_260913_200839.pdf (30.4 MB)

[source4] already present: /Users/mgtimber/CV26/RAG11/stage1_eda_input/source4/Advanced_Nutrition_and_Human_Metabolism.pdf (2.5 MB)
[source3] already present: /Users/mgtimber/CV26/RAG11/stage1_eda_input/source3/Nutrition-Science-and-Everyday-Application-1773787282.pdf (211.2 MB)
[source6] already present: /Users/mgtimber/CV26/RAG11/stage1_eda_input/source6/Clark N. - Nancy Clark's Food Guide for New Runners. Getting It Right from the Start - 2009.pdf (8.4 MB)
[source5] already present: /Users/mgtimber/CV26/RAG11/stage1_eda_input/source5/Clark N. - Nancy Clark's Food Guide for New Runners. Getting It Right from the Start - 2009.pdf (8.4 MB)
[source8] already present: /Users/mgtimber/CV26/RAG11/stage1_eda_input/source8/Krauses_Food_and_the_Nutrit

**Cell #07**

## Write source rows -- ready for `rag11_data_sources`

One row per source, in the same `rowGUID` / `rowOwnerGUID` / `rowParentGUID` /
`orderInList` / `rowJSON` shape as the parent/child chunk tables. A source sits
at the top of the hierarchy, so it owns itself (`rowOwnerGUID == rowGUID`) and
has no parent (`rowParentGUID` is always `None` here). `stage1_2_eda_load_chunks.ipynb`
loads these files into `rag11_data_sources` before the parent/child chunks.


In [4]:
# Cell #08
def build_source_row(source_key: str, order: int) -> dict:
    cfg = SOURCES[source_key]
    row_guid = cfg["row_guid"]
    row_json = {
        "source_key": source_key,
        "drive_folder_url": GOOGLE_DRIVE_SOURCES_FOLDER,
        "drive_folder_id": DRIVE_FOLDER_ID,
        "drive_file_id": cfg["file_id"],
        "filename": cfg["filename"],
        "expected_pages": cfg["expected_pages"],
        "structure": cfg["structure"],
        **source_fetch_info[source_key],
    }
    return {
        "rowGUID": row_guid,
        "rowOwnerGUID": row_guid,   # a source is its own tree's root/owner
        "rowParentGUID": None,      # sources sit above everything -- no parent
        "orderInList": order,
        "rowJSON": row_json,
    }


# Clear stale manifest rows from a previous run (e.g. a source that no
# longer exists in the Drive folder) before writing the current ones.
for stale in SOURCES_MANIFEST_DIR.glob("source_row-*.json"):
    stale.unlink()

for order, key in enumerate(SOURCES, start=1):
    row = build_source_row(key, order)
    (SOURCES_MANIFEST_DIR / f"source_row-{order}.json").write_text(
        json.dumps(row, ensure_ascii=False, indent=2), encoding="utf-8"
    )

print(f"Wrote {len(SOURCES)} source row(s) -> {SOURCES_MANIFEST_DIR} "
      "(loaded into rag11_data_sources by stage1_2)")


Wrote 17 source row(s) -> /Users/mgtimber/CV26/RAG11/stage1_eda_output/sources (loaded into rag11_data_sources by stage1_2)


**Cell #09**

## Extract raw per-page text (cached)

Capped by `MAX_NUMBER_OF_PAGES_TO_USE` (set in Config above) so a full
pipeline run across every registered source can be smoke-tested in
minutes -- set it to `None` before trusting a run's actual output.

In [5]:
# Cell #10
def extract_pages(source_key: str) -> list[str]:
    """
    Return a list of per-page plain text (index 0 = page 1), one entry per
    REAL page in the PDF regardless of MAX_NUMBER_OF_PAGES_TO_USE (so later
    page-number arithmetic elsewhere is never off) -- extracted once and
    cached so re-running later cells does not reparse the whole PDF.

    When MAX_NUMBER_OF_PAGES_TO_USE is set, only that many leading pages
    are actually run through get_text() (the expensive part on a
    1000+ page book); every later page is stored as "" instead. A
    section whose start_page falls past the cap ends up with empty text --
    expected/fine for a fast code-path smoke run, not something to mistake
    for a real per-source verification. Cached to a cap-specific filename
    (_cache_pages.json vs _cache_pages_max100.json) so a fast capped run and
    a full run never silently reuse each other's incomplete/complete cache.
    """
    cache_suffix = f"_max{MAX_NUMBER_OF_PAGES_TO_USE}" if MAX_NUMBER_OF_PAGES_TO_USE is not None else ""
    cache_path = OUTPUT_ROOT / source_key / f"_cache_pages{cache_suffix}.json"
    if cache_path.exists():
        return json.loads(cache_path.read_text(encoding="utf-8"))

    path = downloaded_paths[source_key]
    pages = []
    with fitz.open(path) as doc:
        limit = float("inf") if MAX_NUMBER_OF_PAGES_TO_USE is None else MAX_NUMBER_OF_PAGES_TO_USE
        for i, page in enumerate(doc):
            # PyMuPDF occasionally surfaces a literal NUL codepoint (\u0000)
            # from a malformed embedded font/content stream in the source
            # PDF (seen on source4). Postgres text/jsonb columns can never
            # store \u0000 -- APIError 22P05 'unsupported Unicode escape
            # sequence' -- so it has to be stripped before this text is
            # cached or chunked, not just before the eventual Supabase
            # upsert, or every downstream file (and the page-text cache
            # itself) inherits it and stage1_9's verify step then flags a
            # permanent local-vs-Supabase mismatch on every affected chunk.
            page_text = page.get_text("text") if i < limit else ""
            pages.append(page_text.replace("\x00", ""))

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    cache_path.write_text(json.dumps(pages, ensure_ascii=False), encoding="utf-8")
    n_extracted = min(len(pages), int(limit)) if limit != float("inf") else len(pages)
    print(f"[{source_key}] cached {len(pages)} page(s), text extracted for "
          f"{n_extracted} of them -> {cache_path}")
    return pages


pages_by_source = {key: extract_pages(key) for key in SOURCES}


**Cell #11**

## Section-detection algorithms (per source)

Each source's own algorithm -- native-outline parsing, TOC/regex parsing,
whatever its particular PDF needs -- lives in `./stage1_1_eda_packages/` as
one Python module per source, matched by filename (see that package's
`__init__.py` docstring for how to add a new one). This notebook only
imports the registry; it doesn't define any of the detection logic itself.


In [6]:
# Cell #12
from stage1_1_eda_packages import FILENAME_PROCESSORS


**Cell #13**

## Assemble sections for all sources

In [7]:
# Cell #14
# Route each Drive-discovered file to the right section-detection algorithm
# by FILENAME (stable identity), not by its source1/2/3 slot (which now
# depends on whatever the Drive folder listing/KNOWN_SOURCE_EDA_META order
# assigns it -- see the config cell above). FILENAME_PROCESSORS is imported
# from ./stage1_1_eda_packages/ above.
sections_by_source = {}
for key, cfg in SOURCES.items():
    processor = FILENAME_PROCESSORS.get(cfg["filename"])
    if processor is None:
        raise NotImplementedError(
            f"No section-detection logic registered for '{cfg['filename']}' "
            f"({key}). The Drive folder has a file stage1_1 doesn't know how "
            "to chunk yet -- add a sourceN_<slug>.py module to "
            "./stage1_1_eda_packages/ (an extract_sections(source_key, "
            "pdf_path, pages) function plus FILENAME/EXPECTED_PAGES/STRUCTURE), "
            "and register it in that package's __init__.py."
        )
    sections_by_source[key] = processor(key, downloaded_paths[key], pages_by_source[key])


[source1] 267 sections from outline (levels 2-2)
[source1] 133 sections after dropping attribution bookmarks (brief expects ~140-150)
[source2] TOC pages 6-11 -> 136 sections (brief guessed ~212; this book's real TOC has 20 chapters x ~5-7 numbered/named subsections plus front/back matter -- 136 measured against the actual PDF, verified with zero page-range gaps or overlaps)
[source3] 91 sections from outline (levels 1-2)
[source3] flagged 7 page(s) as possible H5P content gaps (brief mentions ~62): [28, 34, 36, 47, 57, 90, 98]
[source4] 14 chapter-summary sections (chapters 1-14 of 14 listed) + 1 back-matter section(s): ['quotes']
[source4] 6 chapter(s) flagged with a Critical Thinking callout
[source5] 22 sections from outline (dropped 4 boilerplate + 4 part-divider entries)
[source6] 22 sections from outline (dropped 4 boilerplate + 4 part-divider entries)
[source7] 95 sections from outline (levels 1-2)
[source7] 94 sections from outline (levels 1-2), dropped 1 boilerplate entry/ent

**Cell #15**

## Chunking — parent (full section) + child (~300-500 tokens, 10-15% overlap by default)

Child-chunk boundaries are computed on token counts (via `tiktoken`) by
default, and each child chunk is prefixed with a short contextual header
naming its source/section/page range before embedding, per the brief's
recommendation.

A source module can opt out of the fixed-token split for a given section
by setting that section dict's `"children"` key to its own pre-split list
of `{"text": ..., "chunk_type": ...}` dicts (`chunk_type` defaults to
`"prose"`) -- see `child_pieces_for_section()` below. This is how
source15/source16 (encyclopedia-style references, one article/entry per
parent) split each parent into its real named subsections instead of
arbitrary token windows, and emit extracted tables as their own
`chunk_type: "table"` children.


In [8]:
# Cell #16
import tiktoken

_ENC = tiktoken.get_encoding("cl100k_base")


def token_len(text: str) -> int:
    return len(_ENC.encode(text))


def build_child_chunks(text: str, target_tokens: int = 400, overlap_pct: float = 0.125) -> list[str]:
    tokens = _ENC.encode(text)
    if not tokens:
        return []
    step = max(1, int(target_tokens * (1 - overlap_pct)))
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + target_tokens, len(tokens))
        chunks.append(_ENC.decode(tokens[start:end]))
        if end == len(tokens):
            break
        start += step
    return chunks


def child_pieces_for_section(sec: dict) -> list[dict]:
    """
    Return this section's children as {"text": ..., "chunk_type": ...} dicts.

    Most sources have no opinion about how their children should be split,
    so the default path below (fixed ~300-500 token windows with overlap)
    still applies to them -- every child gets chunk_type "prose". A source
    module can instead pre-split a section into semantically real units --
    named subsections, a "Key points" block, extracted tables -- by setting
    sec["children"] to its own list of {"text": ..., "chunk_type": ...}
    dicts (chunk_type defaults to "prose" when omitted, and an empty/blank
    text is dropped); write_chunks() then uses those verbatim instead of
    re-splitting sec["text"] by token count. See
    stage1_1_eda_packages/source15_encyclopedia_human_nutrition.py and
    source16_encyclopedia_of_foods.py for real examples (per-article named
    subsections plus separately-typed "table" children, with References/
    Further Reading excluded entirely so they're never embedded).
    """
    custom = sec.get("children")
    if custom is not None:
        return [
            {"text": piece["text"], "chunk_type": piece.get("chunk_type", "prose")}
            for piece in custom
            if piece.get("text", "").strip()
        ]
    return [{"text": t, "chunk_type": "prose"} for t in build_child_chunks(sec["text"])]


def contextual_header(source_key: str, section: dict) -> str:
    cfg = SOURCES[source_key]
    filename = cfg["filename"]
    title = section["title"]
    start_page = section["start_page"] + 1
    end_page = section["end_page"] + 1
    return f"[Source: {filename} | Section: {title} | Pages {start_page}-{end_page}]"


**Cell #17**

## Write output — `parent_chunk-N.json` / `child_chunk-parentN-chunkM.json`

In [9]:
# Cell #18
def write_chunks(source_key: str) -> None:
    sections = sections_by_source[source_key]
    out_dir = OUTPUT_ROOT / source_key
    out_dir.mkdir(parents=True, exist_ok=True)

    # Clear stale output from a previous run before writing new files. Without
    # this, a run that produces FEWER sections than a prior run leaves old
    # higher-numbered parent_chunk-N.json / child_chunk-parentN-chunkM.json
    # files behind (only the first N are overwritten), which then shows up as
    # bogus "orphaned" rows in stage1_9_eda_verify_all_data.ipynb.
    for stale in out_dir.glob("parent_chunk-*.json"):
        stale.unlink()
    for stale in out_dir.glob("child_chunk-parent*-chunk*.json"):
        stale.unlink()

    # This source's rag11_data_sources rowGUID -- denormalized onto every
    # parent/child row below so stage1_2 can use it directly as
    # rowOwnerGUID (a uuid FK onto rag11_data_sources) instead of the plain
    # 'source1'/'source2' text key the schema used before.
    source_row_guid = SOURCES[source_key]["row_guid"]

    n_parents = 0
    n_children = 0
    for p_idx, sec in enumerate(sections, start=1):
        parent = {
            "parent_id": f"{source_key}-p{p_idx}",
            "source": SOURCES[source_key]["filename"],
            "source_key": source_key,
            "source_row_guid": source_row_guid,
            "title": sec["title"],
            "level": sec.get("level"),
            "start_page": sec["start_page"],
            "end_page": sec["end_page"],
            "block_type": sec.get("block_type", []),
            "text": sec["text"],
        }
        (out_dir / f"parent_chunk-{p_idx}.json").write_text(
            json.dumps(parent, ensure_ascii=False, indent=2), encoding="utf-8"
        )
        n_parents += 1

        header = contextual_header(source_key, sec)
        for c_idx, piece in enumerate(child_pieces_for_section(sec), start=1):
            child_text = piece["text"]
            child = {
                "child_id": f"{source_key}-p{p_idx}-c{c_idx}",
                "parent_id": f"{source_key}-p{p_idx}",
                "source_key": source_key,
                "source_row_guid": source_row_guid,
                "chunk_type": piece["chunk_type"],
                "text": f"{header}\n\n{child_text}",
                "token_count": token_len(child_text),
            }
            (out_dir / f"child_chunk-parent{p_idx}-chunk{c_idx}.json").write_text(
                json.dumps(child, ensure_ascii=False, indent=2), encoding="utf-8"
            )
            n_children += 1

    print(f"[{source_key}] wrote {n_parents} parent chunk(s), {n_children} child chunk(s) -> {out_dir}")


for key in SOURCES:
    write_chunks(key)


[source1] wrote 133 parent chunk(s), 8 child chunk(s) -> /Users/mgtimber/CV26/RAG11/stage1_eda_output/source1
[source2] wrote 136 parent chunk(s), 184 child chunk(s) -> /Users/mgtimber/CV26/RAG11/stage1_eda_output/source2
[source3] wrote 91 parent chunk(s), 185 child chunk(s) -> /Users/mgtimber/CV26/RAG11/stage1_eda_output/source3
[source4] wrote 15 parent chunk(s), 46 child chunk(s) -> /Users/mgtimber/CV26/RAG11/stage1_eda_output/source4
[source5] wrote 22 parent chunk(s), 106 child chunk(s) -> /Users/mgtimber/CV26/RAG11/stage1_eda_output/source5
[source6] wrote 22 parent chunk(s), 106 child chunk(s) -> /Users/mgtimber/CV26/RAG11/stage1_eda_output/source6
[source7] wrote 94 parent chunk(s), 75 child chunk(s) -> /Users/mgtimber/CV26/RAG11/stage1_eda_output/source7
[source8] wrote 97 parent chunk(s), 294 child chunk(s) -> /Users/mgtimber/CV26/RAG11/stage1_eda_output/source8
[source9] wrote 35 parent chunk(s), 173 child chunk(s) -> /Users/mgtimber/CV26/RAG11/stage1_eda_output/source9
[so

**Cell #19**

## Next steps (things to tune after this first run)

- **Tuning an existing source**: each source's detection logic and its
  expected_pages/structure metadata live together in
  `./stage1_1_eda_packages/sourceN_<slug>.py`. Tune the regex/outline-level
  constants at the top of that module (e.g. `min_level`/`max_level` for an
  outline-based source, `SECTION_HEADING_RE`/`CALLOUT_PATTERNS` for a
  regex-based one) and re-run this notebook -- nothing here needs to change.
- **New sources**: dropping another PDF into the shared
  `GOOGLE_DRIVE_SOURCES_FOLDER` Drive folder is enough for it to be picked up
  as `source5`/etc next run -- but it still needs a
  `./stage1_1_eda_packages/sourceN_<slug>.py` module (an
  `extract_sections(source_key, pdf_path, pages)` function plus
  `FILENAME`/`EXPECTED_PAGES`/`STRUCTURE`) registered in that package's
  `__init__.py`, or the "Assemble sections" cell raises `NotImplementedError`
  naming the file. `stage1_1_eda_packages/source4_advanced_nutrition_human_metabolism.py`
  is a worked example of adding one from scratch, including a document that
  turned out to be a condensed chapter-by-chapter summary rather than the
  full textbook, with extra back-matter zones (quotes/Q&A/quiz) folded in as
  their own sections.
- Once boundaries look right, `stage1_2_eda_load_chunks.ipynb` loads this
  notebook's `./stage1_eda_output/sources/source_row-N.json` files into
  `rag11_data_sources`, then the chunk files into `rag11_chunks_parent_table` /
  `rag11_chunks_child_table` -- each parent/child row's `rowOwnerGUID` is now
  the owning source's `rag11_data_sources` rowGUID (see
  `sql/create_sql_tables.sql`), not the old `'source1'`/`'source2'` text.


In [11]:
%%sql
SELECT 
    rowJSON.source_key AS source,
    rowJSON.filename AS filename,
    rowJSON.actual_page_count AS pages
FROM read_json_auto('stage1_eda_output/sources/*.json')
ORDER BY orderInList
-- Cell #20

,source,filename,pages
0,source1,human-nutrition-text.pdf,1208
1,source2,Nutrition_for_Nurses-WEB_260913_200839.pdf,513
2,source3,Nutrition-Science-and-Everyday-Application-177...,649
3,source4,Advanced_Nutrition_and_Human_Metabolism.pdf,181
4,source5,Clark N. - Nancy Clark's Food Guide for New Ru...,161
5,source6,Clark N. - Nancy Clark's Food Guide for New Ru...,161
6,source7,Intuitive_Eating_A_Revolutionary_Program_that_...,240
7,source8,Krauses_Food_and_the_Nutrition_Care_Process.pdf,1159
8,source9,Medical_Nutrition_and_Disease_A_Case_Based_App...,402
9,source10,Sports-Nutrition-Laboratory-Manual-Mary-P.-Mil...,87
